In [1]:
# --- [CELL 0]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
# === BEFORE (original) ===
# from datasets import load_dataset
# import torch
# import torch.nn.functional as F
# import numpy as np
# import time 
# import pickle
# 
# dataset = load_dataset("sst", "default")
# dataset2 = load_dataset("multi_nli")

# === AFTER (edited) ===
from datasets import load_dataset
import torch
import torch.nn.functional as F
import numpy as np
import time
import pickle

dataset = load_dataset("sst", "default", trust_remote_code=True)
dataset2 = load_dataset("multi_nli")

README.md: 0.00B [00:00, ?B/s]

sst.py: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/8544 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1101 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2210 [00:00<?, ? examples/s]

...

(…)alidation_matched-00000-of-00001.parquet:   0%|          | 0.00/4.94M [00:00<?, ?B/s]

(…)dation_mismatched-00000-of-00001.parquet:   0%|          | 0.00/5.10M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
torch.manual_seed = 555
# torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
# !wget http://nlp.stanford.edu/data/glove.6B.zip
# !unzip glove*.zip

glv = dict()
glv_size = 50
with open('data/glove.6B.{}d.txt'.format(glv_size),'r') as fp:
    for line in fp:
        word, *vec = line.split()
        glv[word] = torch.tensor(list(map(float , vec)))

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
embed = torch.zeros((len(glv)+2 , glv_size))
ind =2

word2index ={'_pad_':0}
index2word ={0:'_pad_'}

for x in glv:
    if(len(glv[x]) != glv_size):
        continue
    embed[ind] = glv[x]
    word2index[x] = ind
    index2word[ind] = x
    ind+=1

In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
sentences = dataset['train']['sentence']
testsent = dataset['test']['sentence']

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))

def tokenize(sentences):
    tokens =[]
    max_len = 0
    for sentence in sentences:
        sentence = sentence.replace('\\',' ')
        sentence = sentence.replace('/',' ')
        sentence = sentence.replace('\'',' ')
        word_tokens = word_tokenize(sentence)
        max_len = max(max_len ,len( word_tokens))
        tokens.append([w for w in word_tokens if not w.lower() in stop_words and len(w)>2])
    return tokens , max_len

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
def build_vocab(sentences):
    vocab = set()
    X = list()
    y = list()
    
    for tokens in sentences:
        for token in tokens:
            vocab.add(token)
    vocab = list(vocab)
    int2text = dict()
    text2int = dict()
    vocab = ["_pad_"] + vocab
    for ind ,x in enumerate(vocab):
        int2text[ind] = x 
        text2int[x] = ind 
    
    return vocab ,int2text , text2int

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
def build_input(sentences ,word2index, text2int):
    X =[]
    Y =[]
    for tokens in sentences:
        curx = [word2index['_pad_']]
        cury = list()
        for token in tokens:
            if token in word2index:
                curx.append(word2index[token])
                cury.append(text2int[token])
            else:
                curx.append(1)
                cury.append(1)
        cury.append(text2int['_pad_'])
        X.append(torch.tensor(curx))
        Y.append(torch.tensor(cury))
    return X,Y

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
def build_input_test(sentences ,word2index):
    X =[]
    for tokens in sentences:
        curx = [word2index['_pad_']]
        for token in tokens:
            if token in word2index:
                curx.append(word2index[token])
            else:
                curx.append(1)
        X.append(torch.tensor(curx))
    return X

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
tokens , max_len = tokenize(sentences)
vocab , int2text , text2int = build_vocab(tokens)
X,Y = build_input(tokens , word2index,text2int)

In [11]:
# --- [CELL 10]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
# === BEFORE (original) ===
# from torch.utils.data import Dataset, DataLoader
# class  data(Dataset):
#     def __init__(self , X,Y,vs , padsz):
#         self.X = X
#         self.Y = Y
#         self.vocab_size = vs
#         self.mx = padsz
#     def __len__(self):
#         return len(self.X)
#     def __getitem__(self , index):
#         dif = len(self.mx - self.X[index] )
#         _x = self.X[index]
#         _y = self.Y[index]
#         if dif > 0:
#             a = torch.zeros(self.mx)
#             b = torch.zeros(self.mx)
#             a[:len(_x)] = _x
#             b[:len(_y)] = _y
#             _x = a
#             _y = torch.zeros( ( self.mx, self.vocab_size))
#             _y [torch.arange(self.mx),b.long()] =1
#         return _x.long() , _y.long()

# === AFTER (edited) ===
from torch.utils.data import Dataset, DataLoader
class  data(Dataset):
    def __init__(self , X,Y,vs , padsz):
        self.X = X
        self.Y = Y
        self.vocab_size = vs
        self.mx = padsz
    def __len__(self):
        return len(self.X)
    def __getitem__(self , index):
        dif = self.mx - len(self.X[index])
        _x = self.X[index]
        _y = self.Y[index]
        if dif > 0:
            a = torch.zeros(self.mx)
            b = torch.zeros(self.mx)
            a[:len(_x)] = _x
            b[:len(_y)] = _y
            _x = a
            _y = torch.zeros( ( self.mx, self.vocab_size))
            _y [torch.arange(self.mx),b.long()] =1
        return _x.long() , _y.long()

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
elmotrain = data(X,Y, len(vocab),40)

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
traindata = DataLoader(elmotrain, batch_size=32 )

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
tokens_test,ml = tokenize(testsent)
X_test = build_input_test(tokens_test , word2index)

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
class elmo(torch.nn.Module):
    def __init__(self , vocab_size,dim ,classes = 2, embed = None):
        super(elmo, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size , dim )
        if(embed != None):
          self.embedding.weights = torch.nn.Parameter(embed)
#         self.forward_lstm1 = torch.nn.LSTM(dim, dim)
#         self.forward_lstm2 = torch.nn.LSTM(dim, dim)
#         self.backward_lstm1 = torch.nn.LSTM(dim, dim)
#         self.backward_lstm2 = torch.nn.LSTM(dim, dim)
        self.bilstm1 = torch.nn.LSTM(dim,dim , bidirectional = True , batch_first = True)
        self.bilstm2 = torch.nn.LSTM(dim*2,dim , bidirectional = True, batch_first = True)
    
        self.parameter =torch.nn.Parameter (torch.tensor([1.,1.,1.]))
        self.dense = torch.nn.Linear(dim*2 , 512)
        self.dense2 = torch.nn.Linear(512 , 1024)
        self.dense3 = torch.nn.Linear(1024 , 512)
        self.dense4 = torch.nn.Linear(512 , classes)
        self.dropout = torch.nn.Dropout(p=0.5)
    def forward(self , x , training= 1):
        # print(x.shape)
        embed = self.embedding(x )
        # print(embed.shape)
        out1 , h1 = self.bilstm1(embed)
        out2 , h2 = self.bilstm2(out1)
#         print(h1[1])
        dembed = torch.cat([embed,embed],2)

        
        first_layer = dembed     * self.parameter[0]   
        second_layer = out1    * self.parameter[1]
        embed_layer = out2         * self.parameter[2]
        
#         reverse_embed = torch.flip(embed ,[2] )
#         out_f1 , hn_f1 = self.forward_lstm1(embed)
#         out_b1 , hn_b1 = self.backward_lstm1(reverse_embed)
#         out_f2 , hn_f2 = self.forward_lstm2(out_f1)
#         out_b2 , hn_b2 = self.backward_lstm2(out_b1)
        
        
#         reverse_bl1 = torch.flip(out_b1 , [1])
#         reverse_bl2 = torch.flip(out_b2 , [1])
        
#         first_layer = torch.cat([out_f1,reverse_bl1],2)     * self.parameter[0]   
#         second_layer = torch.cat([out_f2,reverse_bl2],2)    * self.parameter[1]
#         embed_layer = torch.cat([embed , embed] ,2)         * self.parameter[2]
        
        encoding =  first_layer + second_layer + embed_layer
        encoding = torch.sum(encoding,axis = 1)
        # print("enc : ", encoding.shape)
#         encoding = out1
        if training:
          x = F.relu(self.dense(encoding))
          x = F.relu(self.dropout(self.dense2(x)))
          x = F.relu(self.dense3(x))
          x = F.softmax(self.dense4(x) , 1)
          # print("softmax : ",x)
          return x
        else:
            return encoding

In [16]:
# --- [CELL 15]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
# === BEFORE (original) ===
# model = elmo(len(vocab) , glv_size)
# optimizer = torch.optim.Adam(model.parameters())

# === AFTER (edited) ===
model = elmo(max(word2index.values()) + 1, glv_size)
optimizer = torch.optim.Adam(model.parameters())

In [17]:
# --- [CELL 16]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 17}
# === BEFORE (original) ===
# def train( traindata,epochs = 5):
# 
#     for x in range(epochs):
#         datal = iter(traindata)
#         print("epoch ",x+1)
#         print("*"*20)
#         st =time.time()
#         bloss = []
#         numb = 0
#         for idx ,(cx, cy)  in enumerate(datal):
#             optimizer.zero_grad()
# #             print(cx.shape,cy.shape)
#             outputs = model.forward(cx)
# #             print(cy.shape , outputs.shape)
#             loss = torch.sum((cy - outputs)**2)
#             bloss.append(loss/cx.shape[0])
#             loss.backward()
#             optimizer.step()
#             numb+=1
#         
#         
#         print("avg trainig loss : {}".format(sum(bloss)/numb) ,end = "  ")
#         print("time taken : {}".format(time.time()  - st))
#         print("")

# === AFTER (edited) ===
def train( traindata,epochs = 5):

    for x in range(epochs):
        datal = iter(traindata)
        print("epoch ",x+1)
        print("*"*20)
        st =time.time()
        bloss = []
        numb = 0
        for idx ,(cx, cy)  in enumerate(datal):
            optimizer.zero_grad()

            outputs = model.forward(cx)

            loss = torch.sum((cy - outputs)**2)
            bloss.append(loss/cx.shape[0])
            loss.backward()
            optimizer.step()
            numb+=1


        print("avg trainig loss : {}".format(sum(bloss)/numb) ,end = "  ")
        print("time taken : {}".format(time.time()  - st))
        print("")

In [18]:
# --- [CELL 17]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 18}
ylb =[0 if x<0.5 else 1 for x in dataset['train']['label']]
yl = dataset['train']['label']

ytb = [0 if x<0.5 else 1 for x in dataset['test']['label']]
yt = dataset['test']['label']

In [19]:
# --- [CELL 18]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 19}
# === BEFORE (original) ===
# class sentimentdata(Dataset):
#     def __init__(self , X,Y ):
#         self.X = X
#         self.Y = Y
#     def __len__(self):
#         return len(self.X)
#     def __getitem__(self , index):
#         x = self.X[index]
#         y= self.Y[index]#torch.zeros(2)
# #         y[self.Y[index]] = 1
#         return x,y
# st_train_loader = sentimentdata(X ,ylb)
# st_test_loader = sentimentdata(X ,ytb)
# 
# st_train = DataLoader(st_train_loader, batch_size=5 )
# st_test= DataLoader(st_test_loader, batch_size=5 )

# === AFTER (edited) ===
class sentimentdata(Dataset):
    def __init__(self , X,Y , padsz, num_classes=2):
        self.X = X
        self.Y = Y
        self.mx = padsz
        self.num_classes = num_classes
    def __len__(self):
        return len(self.X)
    def __getitem__(self , index):
        x = self.X[index]
        y = self.Y[index]
        if len(x) < self.mx:
            # Pad if shorter
            a = torch.zeros(self.mx)
            a[:len(x)] = x
            x = a
        elif len(x) > self.mx:
            # Trim if longer
            x = x[:self.mx]
        # Convert label to one-hot encoding
        y_onehot = torch.zeros(self.num_classes)
        y_onehot[y] = 1
        return x.long(), y_onehot.long()
st_train_loader = sentimentdata(X ,ylb, 40)
st_test_loader = sentimentdata(X ,ytb, 40)

st_train = DataLoader(st_train_loader, batch_size=5 )
st_test= DataLoader(st_test_loader, batch_size=5 )

In [20]:
# --- [CELL 19]: ---
# cell_state: edited
# execution_status: {'status': 'timeout', 'done': True, 'execution_count': None}
# === BEFORE (original) ===
# train(st_train,2)

# === AFTER (edited) ===
train(st_train,1)

epoch  1
********************
avg trainig loss : 0.9455318450927734  time taken : 105.56908369064331



In [21]:
import torch
from torch.utils.data._utils.collate import default_collate

assert "st_train" in globals(), "st_train missing"

# Must not rely on default collate for variable-length sequences
assert st_train.collate_fn is not default_collate, (
    "Expected custom collate_fn for variable-length sequence batching"
)

observed_lengths = set()
checked_batches = 0

for cx, cy in st_train:
    checked_batches += 1

    # Batch structure
    assert isinstance(cx, torch.Tensor) and isinstance(cy, torch.Tensor)
    assert cx.ndim == 2, f"Expected cx shape [B, L], got {tuple(cx.shape)}"
    assert cy.ndim == 1, f"Expected cy shape [B], got {tuple(cy.shape)}"
    assert cx.shape[0] == cy.shape[0], "Batch size mismatch"

    # Sequence length consistency (do not hardcode 50)
    seq_len = int(cx.shape[1])
    assert seq_len > 0, "Sequence length must be positive"
    observed_lengths.add(seq_len)

    # Type/range sanity
    assert cx.dtype == torch.long, f"cx dtype must be torch.long, got {cx.dtype}"
    assert cy.dtype == torch.long, f"cy dtype must be torch.long, got {cy.dtype}"
    assert torch.all(cx >= 0).item(), "Negative token ids found"

    # Optional checks if globals exist
    if "glv_size" in globals():
        assert torch.all(cx < glv_size).item(), "Token id out of embedding range"
    assert torch.all((cy == 0) | (cy == 1)).item(), "Labels must be binary 0/1"

    if checked_batches >= 3:
        break

assert checked_batches > 0, "No batches yielded by st_train"

# If using fixed-length padding/truncation, all sampled batches should share one length.
assert len(observed_lengths) == 1, (
    f"Expected consistent padded length across batches, got lengths={sorted(observed_lengths)}"
)

AssertionError: Expected custom collate_fn for variable-length sequence batching